## AI Agent



### Define Agent Workflow

In this example, we create an AI agent that acts like an AI chat assistant.
Please note that this template helps you make an agent that uses 'CHAT' interface. You can modify the interface to 'DEFAULT' if the agent is to act like a form based assistant.


In [ ]:
from abacusai import (
    AgentInterface,
    WorkflowGraph,
    WorkflowGraphEdge,
    WorkflowGraphNode,
    WorkflowNodeInputMapping,
    WorkflowNodeInputSchema,
    WorkflowNodeInputType,
    WorkflowNodeOutputMapping,
    WorkflowNodeOutputSchema,
    WorkflowNodeOutputType,
)

agent_interface = AgentInterface.CHAT

### Workflow Graph
To build an AI Agent, we need to build a workflow graph that contains nodes and edges. The nodes represent a step in the execution flow of the AI Agent, while the edges define the flow of data between the nodes and their order of execution.

**Workflow Graph Node**: Each node is linked to a unique function that is executed when the node is reached during the execution of the workflow graph. In addition to function, - the workflow node is defined by name, input mappings, and output mappings.

**Workflow Graph Edge**: Each edge consists of source node, target node and optional details. The edge can be made conditional by leveraging the optional details field e.g. `{'EXECUTION_CONDITION': 'supply>demand'}`


In this example, we define a simple workflow graph with a single node named 'query_handler' that takes in a user query and returns a response.

#### Query Handler node
This node takes in the user query and leverages llm to stream the AI Assistant response to the user. 

You can update below function to do any complex processing as per your use case requirements.

In [ ]:
def query_handler(nlp_query):
    from abacusai import ApiClient
    client = ApiClient()

    output = 'This is a placeholder response. Please update the query_handler function to return a valid response.'
    # Uncomment below line to get response from LLM call
    # output = client.evaluate_prompt(prompt=nlp_query, system_message=f'Please respond to the user query like an AI Assistant.').content
    return output

In [ ]:
query_handler_node = WorkflowGraphNode(
   name='query_handler',
   function=query_handler,
   input_mappings=[
       WorkflowNodeInputMapping(name='nlp_query', variable_type=WorkflowNodeInputType.USER_INPUT)
   ],
   output_mappings=[]
)

workflow_graph = WorkflowGraph(
    nodes=[query_handler_node],
    edges=[]
)

#### [Optional] Test Agent
An agent function can be directly called upon for testing before registration and deployment by just calling the linked function with appropriate inputs.

In [ ]:
query_handler('Give me information about Apple Vision Pro')

#### Advanced configurations
These are optional configurations that only need to be used for advanced use cases. You don't need to make changes here for most of the use cases.


In [ ]:
# We already support popular packages such as pandas and numpy, so there's no need to include them in package_requirements unless you require a specific version.
package_requirements = [] # e.g. ['csv2graph==2.0.1']
included_modules = []  # list of custom-modules created on the platform to include for this agent
agent_connectors = {}  # dict of connectors required for the agent, e.g. {'jiraauth': ['read:jira-user']}
description = ''''''
memory = 14
evaluation_feature_group_id = None
initialize_function = None

### Register Agent
Note that the associated code is read-only, please leverage the cells above to make edits to the agent function.


In [ ]:
from abacusai import ApiClient
client = ApiClient()
if 'workflow_graph' not in vars():
    raise Exception('Please define agent workflow graph.')
initialize_function_name = None
if initialize_function:
    initialize_function_name = initialize_function.__name__
agent = client.create_agent(project_id='ff5ac4dec', name='example_agent', package_requirements=package_requirements, description=description, memory=memory, evaluation_feature_group_id=evaluation_feature_group_id, workflow_graph=workflow_graph, agent_interface=agent_interface, included_modules=included_modules, agent_connectors=agent_connectors, initialize_function_name=initialize_function_name, initialize_function_code=initialize_function)
agent.wait_for_publish()
agent

### [Optional] Deploy Agent
Agent needs to be deployed before used for querying. An agent could be deployed using the following APIs.


In [ ]:
from abacusai import ApiClient
client = ApiClient()
project = client.describe_project('ff5ac4dec')
deployment_token = client.create_deployment_token(project).deployment_token
if 'agent' not in vars():
    raise Exception('Please register the agent first!!')
deployment = client.create_deployment(model_id=agent.agent_id)
deployment.wait_for_deployment()